<a href="https://colab.research.google.com/github/Evaline2001/Evaline2001/blob/main/INVENTORY_DASHBOARD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask pandas
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb


In [ ]:
import os

project_name = "bittiner_flask_dashboard"
os.makedirs(f"{project_name}/templates", exist_ok=True)
os.makedirs(f"{project_name}/uploads", exist_ok=True)

# ---------- app.py ----------
app_code = """
from flask import Flask, render_template, request, jsonify, redirect, url_for
import pandas as pd
import os

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'bittiner_flask_dashboard/uploads'
ALLOWED_EXTENSIONS = {'xlsx', 'xls'}

# Default dataset
data = {
    "Week (Date)": ["06/09/2025", "13/09/2025", "20/09/2025", "27/09/2025"],
    "Revenue (Ksh)": [125000, 142500, 158000, 176800],
    "Units Sold": [230, 280, 310, 350],
    "Social Media Conversion Rate (%)": [2.4, 2.8, 3.0, 3.2],
    "Social Media Following": [120000, 160000, 210000, 245000],
    "Weekly Production (Units)": [250, 300, 320, 350],
    "Inventory Levels": [500, 450, 420, 400],
    "Fabric Quality Checks": [47, 50, 49, 52],
}
df = pd.DataFrame(data)


def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS


@app.route("/")
def index():
    return render_template("index.html", data=df.to_dict(orient="records"))


@app.route("/upload", methods=["POST"])
def upload_excel():
    global df
    file = request.files.get("file")
    if file and allowed_file(file.filename):
        filepath = os.path.join(app.config["UPLOAD_FOLDER"], file.filename)
        file.save(filepath)
        df = pd.read_excel(filepath)
        return redirect(url_for("index"))
    return "❌ Invalid file format. Please upload an Excel (.xlsx or .xls) file."


@app.route("/add", methods=["POST"])
def add_data():
    global df
    new_row = {
        "Week (Date)": request.form["week"],
        "Revenue (Ksh)": int(request.form["revenue"]),
        "Units Sold": int(request.form["units"]),
        "Social Media Conversion Rate (%)": float(request.form["conversion"]),
        "Social Media Following": int(request.form["followers"]),
        "Weekly Production (Units)": int(request.form["production"]),
        "Inventory Levels": int(request.form["inventory"]),
        "Fabric Quality Checks": int(request.form["quality"]),
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    return jsonify({"status": "success"})


@app.route("/export")
def export_data():
    return df.to_csv(index=False)


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
"""
with open(f"{project_name}/app.py", "w") as f:
    f.write(app_code)


# ---------- index.html ----------
html_code = """
<!DOCTYPE html>
<html>
<head>
  <title>Bittiner Hub Dashboard</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style>
    body { font-family: Arial, sans-serif; margin: 30px; }
    h1 { text-align: center; }
    table { border-collapse: collapse; width: 100%; margin-bottom: 40px; }
    th, td { border: 1px solid #ddd; padding: 8px; text-align: center; }
    th { background-color: #f2f2f2; }
    tr:hover { background-color: #f9f9f9; }
    .upload-box { border: 2px dashed #aaa; padding: 20px; text-align: center; margin-bottom: 30px; border-radius: 10px; background: #f8f9fa; }
    .upload-box:hover { background: #eef3ff; }
    input[type=file] { margin-top: 10px; }
    button { background: #007bff; color: white; border: none; padding: 8px 14px; border-radius: 6px; cursor: pointer; }
    button:hover { background: #0056b3; }
  </style>
</head>
<body>
  <h1>📊 Bittiner Hub – Weekly Dashboard</h1>
  <div class="upload-box" style="border:2px dashed #aaa;padding:20px;text-align:center;margin-bottom:30px;border-radius:10px;">
  <form action="/upload" method="post" enctype="multipart/form-data">
    <h3>📥 Import Excel File</h3>
    <p>Select an Excel (.xlsx or .xls) file to update the dashboard.</p>
    <input type="file" name="file" required>
    <br><br>
    <button type="submit" style="background:#007bff;color:white;border:none;padding:8px 14px;border-radius:6px;cursor:pointer;">Upload & Refresh</button>
  </form>
</div>

<h3>Data Table</h3>


  <h3>Data Table</h3>
  <table>
    <tr>
      {% for key in data[0].keys() %}
        <th>{{ key }}</th>
      {% endfor %}
    </tr>
    {% for row in data %}
    <tr>
      {% for value in row.values() %}
        <td>{{ value }}</td>
      {% endfor %}
    </tr>
    {% endfor %}
  </table>

  <h3>Revenue Trend</h3>
  <div id="revenueChart"></div>

  <script>
    var weeks = {{ data | map(attribute="Week (Date)") | list | safe }};
    var revenue = {{ data | map(attribute="Revenue (Ksh)") | list | safe }};

    var trace = {
      x: weeks,
      y: revenue,
      type: 'scatter',
      mode: 'lines+markers',
      name: 'Revenue',
      line: { color: 'royalblue' }
    };

    Plotly.newPlot('revenueChart', [trace], { title: 'Revenue Over Time' });
  </script>
</body>
</html>
"""
with open(f"{project_name}/templates/index.html", "w") as f:
    f.write(html_code)

print("✅ Project files with Excel import created successfully.")


In [ ]:
# run Flask in background
!nohup python3 bittiner_flask_dashboard/app.py &


In [ ]:
!cloudflared tunnel --url http://localhost:5000 --no-autoupdate


2025-10-07T08:00:46Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-10-07T08:00:46Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-10-07T08:00:49Z INF +--------------------------------------------------------------------------------------------+
2025-10-07T08:00:49Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-10-07T08:00:49Z INF |  https://including-charming-praise-middle.trycloudflar